# 말투 AI — Qwen3-8B 학습 (무료 Colab T4) → 내 PC에서 답장

PC의 GPU(8GB)로는 8B 모델을 학습할 수 없어서, 학습만 Colab에서 하고 결과 **어댑터 파일(약 180MB)** 을 PC로 옮깁니다.
PC 학습 서버가 그 어댑터로 답장을 만들고, 휴대폰은 연결 주소로 대화만 합니다.

1. Colab 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 고른 뒤, 위에서부터 차례로 실행하세요.
2. 학습은 **1.5~2.5시간** 걸립니다. 무료 Colab은 오래 손대지 않으면 끊길 수 있으니 창을 열어 두세요.
3. 마지막에 받은 `persona-8b-….zip` 을 PC에서 **http://localhost:8001** → "Colab에서 학습한 8B 모델 가져오기"로 올리면 끝입니다.

PC에서 학습하는 `qwen3-4b-pc`와 같은 학습 코드(`persona_train.py`)를 씁니다: 긴 대화 단위, 맞장구만 있는 답 줄이기, 답장 길이·질문 힌트.

> ⚠️ 올린 대화는 Google 서버(내 Colab 런타임)로 전송됩니다. 끝나면 마지막 셀로 지우고, 받은 zip은 공개된 곳에 올리지 마세요(대화 내용이 일부 기억될 수 있음).

## 1. 설정

In [ ]:
REPO_REF = "master"        # 아직 master에 합쳐지지 않았다면 "pc-reply-trainer"
TARGET = ""                # 카톡 내보내기(.txt/.csv)를 올릴 때: 따라할 사람 이름(대화에 나온 그대로)
EXCLUDE = []               # 빼고 학습할 참여자(봇 등), 예: ["플레이봇", "채팅봇"]
NAME = ""                  # 앱에 보일 이름(비우면 TARGET). 앱에서 내보낸 .jsonl을 올릴 때는 꼭 입력

## 2. 설치 (약 5분)
PC 학습 서버와 같은 버전으로 설치합니다(Colab에 미리 깔린 torch와 섞이면 unsloth가 안 불러와집니다).
마지막 줄에 `unsloth … GPU Tesla T4`가 나와야 합니다. 오류가 나면 **런타임 → 세션 다시 시작** 후 이 셀부터 다시 실행하세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q "unsloth==2026.10.4" "unsloth_zoo==2026.10.7" "torch==2.14.1" "torchvision==0.29.1" "torchao==0.18.0" \
    "xformers==0.0.35" "triton==3.8.0" "transformers==5.17.0" "trl==1.15.0" "peft==0.21.2" "accelerate==1.15.0" \
    "bitsandbytes==0.50.2" "datasets==4.8.5" sentencepiece "protobuf<5"
!pip uninstall -y -q torchaudio  # Colab's copy is built for its own torch and breaks imports after the upgrade
!rm -rf Talk_AI && git clone -q --depth 1 -b {REPO_REF} https://github.com/jshaa/Talk_AI.git && echo "코드 준비 완료"
!python -c "import unsloth, torch; print('unsloth', unsloth.__version__, '| torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))"

## 3. 대화 올리기
카카오톡 대화 내보내기 파일(.txt/.csv) 또는 앱 ① 탭에서 내려받은 `persona_train.jsonl` 을 고르세요.
전화번호·이메일 등은 자동으로 가려지고, 올린 원본 파일은 바로 지웁니다.

In [ ]:
import collections, json, os, sys
from pathlib import Path
from google.colab import files

sys.path.insert(0, "Talk_AI/training")
import prepare_kakao as pk

uploaded = files.upload()
path = Path(next(iter(uploaded)))
if path.suffix == ".jsonl":
    assert NAME, "앱에서 내보낸 .jsonl은 1번 셀의 NAME에 따라할 이름(앱과 같게)을 넣어 주세요."
    PERSONA, data = NAME, path.read_text(encoding="utf-8")
else:
    fmt, messages, _ = pk.parse_export(pk.read_text(path))
    assert fmt, "카카오톡 내보내기 파일 형식이 아닙니다."
    print("참여자:", collections.Counter(m.speaker for m in messages).most_common())
    assert TARGET, "1번 셀의 TARGET에 위 참여자 중 따라할 이름을 넣고 이 셀을 다시 실행하세요."
    messages = [m for m in messages if m.speaker not in set(EXCLUDE)]
    # PC 답장은 최근 약 3,000자를 보므로 그만큼 긴 대화 단위로 학습합니다(PC의 submit.py와 같음).
    samples = pk.build_samples(messages, TARGET, NAME or None, max_turns=24, max_sample_chars=3000, overlap_turns=6)
    assert samples, f"'{TARGET}'의 답장이 없습니다. 이름을 확인하세요."
    PERSONA, data = NAME or TARGET, pk.to_jsonl(samples)
Path("data.jsonl").write_text(data, encoding="utf-8")
path.unlink()
print(f"따라할 이름: {PERSONA} · 학습 샘플 {sum(1 for l in data.splitlines() if l.strip())}개")

## 4. 학습 (1.5~2.5시간)

In [ ]:
!python Talk_AI/training/server/persona_train.py --data data.jsonl --out out --base qwen3-8b-pc --delete-data

## 5. 바로 시험해 보기 (선택, 약 3분)
PC와 같은 방식(답장 후보 3개 중 고르기)으로 몇 마디 주고받아 봅니다.

In [ ]:
import torch
sys.path.insert(0, "Talk_AI/training/server")
import chat_worker as cw
from unsloth import FastLanguageModel

system = pk.system_prompt(PERSONA)  # 학습 데이터·앱과 같은 시스템 프롬프트
model, tok = FastLanguageModel.from_pretrained(model_name="out/adapter", max_seq_length=3072, load_in_4bit=True)
FastLanguageModel.for_inference(model)
end = tok.convert_tokens_to_ids("<|im_end|>")

def generate(prompt, n, temperature, max_new):
    ids = tok(prompt, add_special_tokens=False, return_tensors="pt").input_ids.to("cuda")
    with torch.inference_mode():
        out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new, do_sample=True,
                             temperature=temperature, top_p=0.92, top_k=50, min_p=0.05, repetition_penalty=1.12,
                             num_return_sequences=n, eos_token_id=end, pad_token_id=end)
    return [tok.decode(o[ids.shape[1]:], skip_special_tokens=True) for o in out]

count = lambda s: len(tok(s, add_special_tokens=False).input_ids)
msgs = [{"role": "system", "content": system}]
for text in ["안뇽 뭐해?", "나 방금 퇴근했어 ㅠㅠ", "오늘 팀장님한테 혼났어", "아까 내가 누구한테 혼났다고 했지?"]:
    msgs.append({"role": "user", "content": text})
    reply = cw.respond(msgs, 0.8, generate, count)
    msgs.append({"role": "assistant", "content": reply})
    print("나:", text); print("AI:", reply.replace("\n", " / ")); print()
del model; torch.cuda.empty_cache()

## 6. 어댑터 내려받기 → PC로 가져오기
받은 `persona-8b-….zip` 을 PC로 옮긴 뒤, PC에서 Docker 학습 서버를 켜고 **http://localhost:8001** →
**"Colab에서 학습한 8B 모델 가져오기"** 에서 고르세요. 변환과 8B 기본 모델 다운로드(처음 한 번, 약 5GB)가 끝나면
휴대폰에서 연결 주소를 열어 대화하면 됩니다.

In [ ]:
import zipfile

status = json.loads(Path("out/status.json").read_text(encoding="utf-8"))
assert status["stage"] == "done", status.get("message")
meta = {"base": "qwen3-8b-pc", "persona": PERSONA, "samples": status.get("samples"), "replies": status.get("replies"), "loss": status.get("loss")}
zip_name = f"persona-8b-{PERSONA}.zip"
with zipfile.ZipFile(zip_name, "w") as z:
    z.writestr("meta.json", json.dumps(meta, ensure_ascii=False))
    for name in ("adapter_config.json", "adapter_model.safetensors"):
        z.write(f"out/adapter/{name}", name)
print(zip_name, f"{os.path.getsize(zip_name) / 1e6:.0f} MB")
files.download(zip_name)

## 7. 정리 (개인정보)
다운로드가 끝난 뒤 실행하세요. 런타임 안의 학습 데이터·어댑터를 지웁니다.

In [ ]:
import glob, shutil
for p in ["out", "data.jsonl", *glob.glob("persona-8b-*.zip")]:
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else Path(p).unlink(missing_ok=True)
print("정리 완료")